In [1]:
# workaround for incredibly stupid vscode on snap bug
from os import environ
krb5ccname = environ["KRB5CCNAME"]
if "hostfs" in krb5ccname:
    krb5ccname = krb5ccname.replace("/var/lib/snapd/hostfs", "")
    environ["KRB5CCNAME"] = krb5ccname

In [2]:
import ROOT
from analysis_framework import Dataset
from OptimalObservableHelper import OptimalObservableHelper
from AltSetupHandler import AltSetupHandler

OBJ: TStyle	ildStyle	ILD Style : 0 at: 0x9659b50
OBJ: TStyle	ildStyle	ILD Style : 0 at: 0x9637d80


In [3]:
# CLD
# x_angle = 0.030 # rad
# ILD
x_angle = 0.014 # rad
n_threads = 12
# n_threads = 6
no_rvec = True
# write_outputs = False
write_outputs = True
dataset_path = "data/datasets/reweighted/signal-only.json"
# friend_dataset_path = "data/datasets/truth-reweighted/signal-only.json"
friend_dataset_path = "data/datasets/truth-reweighted-hel/signal-only.json"


In [4]:
ROOT.EnableImplicitMT(n_threads)
# environ["OMP_NUM_THREADS"] = "6"

In [5]:
dataset = Dataset.from_json(dataset_path)
friend_dataset = Dataset.from_json(friend_dataset_path)

In [6]:
analysis = OptimalObservableHelper(dataset, friend_datasets=[friend_dataset])

missing friend for sample: 4f_sw_sl_eLpL_bkg
missing friend for sample: 4f_sw_sl_eLpR_bkg
missing friend for sample: 4f_sw_sl_eRpR_bkg
missing friend for sample: 4f_sw_sl_eRpL_bkg
OBJ: TStyle	ildStyle	ILD Style : 0 at: 0xd3e5020


In [7]:
analysis.init_categories()
# check if we missed any processes
print(analysis.is_complete_categorisation())
signal_category = ["4f_sw_sl_signal"]

True


In [8]:
oo_configs = [
    "g1z_pos_1em08",
    "ka_pos_1em08",
    "la_pos_1em08",
    ]
# analysis.define_optimal_observables_truth(oo_configs, truth_categories=signal_category)
# analysis.define_optimal_observables_truth_averaged(oo_configs, truth_categories=signal_category)
# analysis.define_optimal_observables_reco(oo_configs)
# oo_names = [f"O_{c}" for c in oo_configs]
oo_names = {
    "mlvec_reco_oo": analysis.define_optimal_observables("mlvec_O", ["mlvec_reco_sqme", "wj_mlvec_reco_sqme"], oo_configs, categories=signal_category),
    "reco_oo": analysis.define_optimal_observables("O", ["reco_sqme", "wj_reco_sqme"], oo_configs, categories=signal_category),
    "reco_jm_oo": analysis.define_optimal_observables("jm_O", ["reco_sqme"], oo_configs, categories=signal_category),
    "clean_reco_oo": analysis.define_optimal_observables("clean_O", ["clean_reco_sqme", "wj_clean_reco_sqme"], oo_configs, categories=signal_category),
    "clean_reco_jm_oo": analysis.define_optimal_observables("clean_jm_O", ["clean_reco_sqme"], oo_configs, categories=signal_category),
    "cheat_clean_reco_oo": analysis.define_optimal_observables("cheat_clean_O", ["cheat_clean_reco_sqme", "wj_cheat_clean_reco_sqme"], oo_configs, categories=signal_category),
    "cheat_clean_reco_jm_oo": analysis.define_optimal_observables("cheat_clean_jm_O", ["cheat_clean_reco_sqme"], oo_configs, categories=signal_category),
    "mc_oo": analysis.define_optimal_observables("mc_O", ["mc_sqme"], oo_configs, categories=signal_category),
    "nomb_mc_oo": analysis.define_optimal_observables("nomb_mc_O", ["nomb_mc_sqme"], oo_configs, categories=signal_category),
    "nomb_mc_rlep_oo": analysis.define_optimal_observables("nomb_mc_rlep_O", ["nomb_mc_rlep_sqme"], oo_configs, categories=signal_category),
    "av_mc_oo": analysis.define_optimal_observables("av_mc_O", ["mc_sqme", "wj_mc_sqme"], oo_configs, categories=signal_category),
    "av_nomb_mc_oo": analysis.define_optimal_observables("av_nomb_mc_O", ["nomb_mc_sqme", "wj_nomb_mc_sqme"], oo_configs, categories=signal_category),
    "av_nomb_mc_rlep_oo": analysis.define_optimal_observables("av_nomb_mc_rlep_O", ["nomb_mc_rlep_sqme", "wj_nomb_mc_rlep_sqme"], oo_configs, categories=signal_category),
}

In [9]:
for names in oo_names.values():
    for name in names:
        # create dummy oo where they don't exist yet
        analysis.define_only_on(["4f_sl_bkg"], name, "0.")
# TODO: re-consider what is correct here...
analysis.add_filter("&&".join([f"std::isfinite({oo})" for oo_name in oo_names.values() for oo in oo_name]), "finite OO")
# analysis.add_filter("&&".join([f"abs({oo}) <= 5" for oo in oo_names["reco_oo"]]), "abs(OO) <= 5")
# analysis.add_filter("(iso_lep_lvec + nu_lvec).M() > 0.", "physical nu")
# analysis.add_filter("(iso_lep_lvec + clean_nu_lvec).M() > 0.", "physical clean_nu")
# analysis.add_filter("nu_lvec.E() > 0.", "physical nu")
# analysis.add_filter("clean_nu_lvec.E() > 0.", "physical clean nu")
analysis.book_reports()

In [10]:
analysis.book_histogram_1D("mc_O_g1z_pos_1em08", "mc_O_g1z_pos_1em08", ("", "", 250, -2.5, 2.5), categories=signal_category)
analysis.book_histogram_1D("mc_O_ka_pos_1em08", "mc_O_ka_pos_1em08", ("", "", 250, -2.5, 2.5), categories=signal_category)
analysis.book_histogram_1D("mc_O_la_pos_1em08", "mc_O_la_pos_1em08", ("", "", 250, -2.5, 2.5), categories=signal_category)

In [11]:
alt_setup_handler = AltSetupHandler("""
{
  "SM": {
    "g1z": 1.0,
    "ka": 1.0,
    "la": 0.0
  },
"variations": [
    0.0025,
    -0.0025,
    0.002,
    -0.002,
    0.0015,
    -0.0015,
    0.001,
    -0.001,
    7.5e-04,
    -7.5e-04,
    5e-04,
    -5e-04,
    2.5e-04,
    -2.5e-04,
    1e-04,
    -1e-04,
    1e-05,
    1e-06,
    1e-07,
    1e-08,
    1e-09,
    1e-10,
    1e-11,
    1e-12,
    1e-13,
    1e-14,
    1e-15,
    1e-16
  ]
}
""", mirror=False, combinations=False)
alt_config_names = list(alt_setup_handler.get_alt_setup().keys())

In [ ]:
weight_names = analysis.book_weight_sums(["nominal"] + alt_config_names, categories=signal_category)
# weight_names = analysis.book_weight_sums(["nominal"] + [name for name in alt_config_names if "em03" in name] + [name for name in alt_config_names if "em04" in name], categories=signal_category)

In [13]:
for names in oo_names.values():
    analysis.define_weighted_oo(names, weight_names, categories=signal_category)
    analysis.book_oo_sums(names, weight_names, categories=signal_category)
    analysis.book_oo_matrix(names, categories=signal_category)

In [14]:
%%time
analysis.run()

CPU times: user 52min 28s, sys: 22 s, total: 52min 50s
Wall time: 5min 42s


In [15]:
analysis.print_reports()

         4f_sw_sl_signal               4f_sl_bkg
        10244400 (1e-03)           33239 (2e-02) All
        10236314 (1e-03)           33239 (2e-02) finite OO
                    1.00                    1.00 efficiency



In [16]:
print(alt_config_names)

['g1z_pos_3em03', 'ka_pos_3em03', 'la_pos_3em03', 'g1z_neg_3em03', 'ka_neg_3em03', 'la_neg_3em03', 'g1z_pos_2em03', 'ka_pos_2em03', 'la_pos_2em03', 'g1z_neg_2em03', 'ka_neg_2em03', 'la_neg_2em03', 'g1z_pos_1em03', 'ka_pos_1em03', 'la_pos_1em03', 'g1z_neg_1em03', 'ka_neg_1em03', 'la_neg_1em03', 'g1z_pos_8em04', 'ka_pos_8em04', 'la_pos_8em04', 'g1z_neg_8em04', 'ka_neg_8em04', 'la_neg_8em04', 'g1z_pos_5em04', 'ka_pos_5em04', 'la_pos_5em04', 'g1z_neg_5em04', 'ka_neg_5em04', 'la_neg_5em04', 'g1z_pos_3em04', 'ka_pos_3em04', 'la_pos_3em04', 'g1z_neg_3em04', 'ka_neg_3em04', 'la_neg_3em04', 'g1z_pos_1em04', 'ka_pos_1em04', 'la_pos_1em04', 'g1z_neg_1em04', 'ka_neg_1em04', 'la_neg_1em04', 'g1z_pos_1em05', 'ka_pos_1em05', 'la_pos_1em05', 'g1z_pos_1em06', 'ka_pos_1em06', 'la_pos_1em06', 'g1z_pos_1em07', 'ka_pos_1em07', 'la_pos_1em07', 'g1z_pos_1em08', 'ka_pos_1em08', 'la_pos_1em08', 'g1z_pos_1em09', 'ka_pos_1em09', 'la_pos_1em09', 'g1z_pos_1em10', 'ka_pos_1em10', 'la_pos_1em10', 'g1z_pos_1em11', 'k

In [ ]:
# calculate means etc.
# names = oo_names["mc_oo"]
names = oo_names["mlvec_reco_oo"]
pars = ["g1z", "ka", "la"]

oo_means = analysis.calc_oo_means(names, weight_names, categories=signal_category)


In [31]:
oo_slopes = analysis.get_slopes(names, pars, oo_means, g=1e-4)

In [32]:
oo_slopes.keys()

dict_keys(['mlvec_O_g1z_pos_1em08_g1z', 'mlvec_O_g1z_pos_1em08_ka', 'mlvec_O_g1z_pos_1em08_la', 'mlvec_O_ka_pos_1em08_g1z', 'mlvec_O_ka_pos_1em08_ka', 'mlvec_O_ka_pos_1em08_la', 'mlvec_O_la_pos_1em08_g1z', 'mlvec_O_la_pos_1em08_ka', 'mlvec_O_la_pos_1em08_la'])

In [ ]:
slope_comp_g_orders = list(range(3, 17))
# slope_comp_g_vals = [10**-x for x in slope_comp_g_orders]

slope_val_graphs = {k: ROOT.TGraph() for k in names}

for x in slope_comp_g_orders:
    g = 10**(-x)
    slopes = analysis.get_slopes(names, pars, oo_means, g=g)
    for k, s in slopes.items():
        key =
        slope_val_graphs[k].AddPoint(x, s)

slope_val_canvs = {}
for k, g in slope_val_graphs.items():
    c = ROOT.TCanvas()
    g.SetTitle(k)
    g.Draw("alp")
    c.Draw()
    slope_val_canvs[k] = c

KeyError: 'mlvec_O_g1z_pos_1em08_g1z'

In [18]:
# x_points = [-2e-3, -1.5e-3, -1e-3, -5e-4, 5e-4, 1e-3, 1.5e-3, 2e-3]
x_points = [-1.5e-3, -1e-3, -5e-4, -2.5e-4, -1e-4, 1e-4, 2.5e-4, 5e-4, 1e-3, 1.5e-3]
oo_graphs = analysis.make_slope_graphs(names, pars, x_points, oo_means)

In [27]:
canvases = {}
f_slopes = {}
for k, g in oo_graphs.items():
    c = ROOT.TCanvas()
    g.Draw("alp")
    f = ROOT.TF1(f"f_{k}", f"{oo_slopes[k]} * x + 1.", -0.01, 0.01)
    f.Draw("same")
    # f.Draw()
    f_slopes[k] = f
    c.Draw()
    canvases[k] = c